<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Ingeniería y Ciencia de Datos</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Laboratorio de Procesamiento de Datos</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Módulo 1: Detección de Datos Atípicos</h3>
    </div>
</div>


##  ¿Qué es un dato atípico?

Un dato atípico es una observación que se comporta de forma inusual **respecto a una referencia**. Esa referencia puede ser una variable, una población, un grupo, una vecindad o una relación entre variables.

Puede deberse a:

- un error de captura o medición;
- una unidad distinta o un problema de escala;
- un caso válido pero poco frecuente;
- un subgrupo real de la población;
- una observación influyente para un modelo.

> **Regla de trabajo:** un detector produce candidatos, no veredictos. Antes de eliminar un registro, revisa su contexto, trazabilidad y efecto en el análisis.

### Dos perspectivas

| Perspectiva | Pregunta | Ejemplo |
|---|---|---|
| **Univariada** | ¿Es raro el valor de una variable? | un `bmi` muy alejado del resto |
| **Multivariada** | ¿Es rara la combinación de variables? | edad y colesterol cuya combinación es inusual aunque cada valor aislado no lo sea |

Un mismo registro puede ser normal en una variable y atípico en otra. Por eso el método debe corresponder a la pregunta.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from collections import Counter
from scipy.stats import median_abs_deviation
from sklearn.linear_model import LinearRegression

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
N = 20
x = np.linspace(0, 10, N)
y = 10 + 2 * x + np.random.normal(loc=0, scale=2, size=(N,))
y[0] = 30
y[-1] = 10

df_xy = pd.DataFrame({'x': x, 'y': y})
#distribuciones de x, y
plt.figure(figsize=(6,4))
df_xy.boxplot(figsize=(6,5));
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.show()

In [ ]:
df_xy

In [ ]:
# Relación entre x, y
plt.figure(figsize=(6,4))
plt.plot(x,y,'xr', label='datos')
plt.grid()
plt.show()

## Outliers Univariados

Las metodologías más comunes para detectarlos son:

1. Método del Rango Intercuantílico (IQR)
2. Método de la desviación estándar
3. Método de la puntuación Z, Z-score modificada

#### Método del Rango Intercuantílico (IQR)

El concepto de rango intercuartílico (IQR) se utiliza para construir los diagramas de caja. El IQR es un concepto en estadística que se utiliza para medir la dispersión y variabilidad de los datos dividiendo el conjunto en cuartiles.

Cualquier conjunto de datos u observaciones se divide en cuatro intervalos definidos según los valores de los datos y cómo se comparan con el conjunto completo. Un cuartil es lo que divide los datos en tres puntos y cuatro intervalos.

$IQR$ es la diferencia entre el tercer cuartil y el primer cuartil ($IQR = Q_3 - Q_1$). Los valores atípicos en este caso se definen como las observaciones que están por debajo de ($Q1 − 1.5*IQR$) o el bigote inferior del diagrama de caja, o por encima de ($Q3 + 1.5* IQR$) o el bigote superior. Puede representarse visualmente mediante el diagrama de caja.

In [ ]:
from sklearn.datasets import load_iris

LI=load_iris()
df=pd.DataFrame(LI.data,columns=LI.feature_names)
df.head()

In [ ]:
df.hist()
plt.show()

In [ ]:
df['sepal width (cm)'].values

In [ ]:
q3,q1=np.quantile(df,(0.75,0.25),axis=0)
iqr=q3-q1
iqr

In [ ]:
Li=q1-1.5*iqr
Ls=q3+1.5*iqr
Li

In [ ]:
Ls

In [ ]:
(df.iloc[:,1] < Li[1]) | (df.iloc[:,1] > Ls[1] ) #Outliers para sepal width (cm)

In [ ]:
spel_width_outliers = df[(df.iloc[:,1] < Li[1]) | (df.iloc[:,1] > Ls[1] )]['sepal width (cm)'] #Outliers para sepal length (cm)
spel_width_outliers

In [ ]:
spel_width_woutliers = df[~((df.iloc[:,1] < Li[1]) | (df.iloc[:,1] > Ls[1] ))]['sepal width (cm)']

In [ ]:
spel_width_woutliers.hist()
plt.show()

In [ ]:
def MetodoIQR(df, n, features):
    """Devuelve índices marcados en al menos n variables por la regla IQR."""
    
    return None

In [ ]:
df

In [ ]:
outliers_index = MetodoIQR(df, 1, df.columns)
outliers_index

### Método de la desviación estándar

Si sabemos que la distribución de los valores en la muestra es gaussiana o similar a la gaussiana, podemos usar la desviación estándar de la muestra como un límite para identificar valores atípicos.

La desviación estándar muestra cuánto se dispersan los puntos de datos individuales respecto a la media. Si la distribución de los datos es normal entonces:
* El 68% de los valores de los datos se encuentran dentro de una desviación estándar de la media
* El 95% están dentro de dos desviaciones estándar
* El 99.7% se encuentran dentro de tres desviaciones estándar.

Dependiendo de la especificación establecida, ya sea a 2 o 3 veces la desviación estándar, podemos detectar y eliminar valores atípicos del conjunto de datos.

Este método puede fallar en la detección de valores atípicos porque los valores atípicos aumentan la desviación estándar. Cuanto más extremo sea el valor atípico, más se ve afectada la desviación estándar.

<img src="Figures/Standard_deviation_diagram.svg" width="800" height="800">


In [ ]:
# El conjunto de datos contiene transacciones realizadas con tarjetas de crédito. Contiene únicamente variables de entrada numéricas que son el resultado de una transformación PCA.
df_credit = pd.read_csv('Data/creditcard.csv') 
df_credit.head()

In [ ]:
df_credit.columns[:-1]

In [ ]:
feature_list = df_credit.columns[:-1].to_list()

In [ ]:
#Ploteamos los diagramas de caja de algunas características
fig, axes = plt.subplots(nrows=2, ncols=3,figsize=(12,8))
fig.suptitle('Diagrama de caja Características vs Class\n', size = 18)

sns.boxplot(ax=axes[0, 0], data=df_credit, x='Class', y='V17')
axes[0,0].set_title("V17");

sns.boxplot(ax=axes[0, 1], data=df_credit, x='Class', y='V10')
axes[0,1].set_title("V10");

sns.boxplot(ax=axes[0, 2], data=df_credit, x='Class', y='V12')
axes[0,2].set_title("V12");

sns.boxplot(ax=axes[1, 0], data=df_credit, x='Class', y='V16')
axes[1,0].set_title("V16");

sns.boxplot(ax=axes[1, 1], data=df_credit, x='Class', y='V14')
axes[1,1].set_title("V14");

sns.boxplot(ax=axes[1, 2], data=df_credit, x='Class', y='V3')
axes[1,2].set_title("V3");

plt.tight_layout()
plt.show()

In [ ]:
df_credit.columns

In [ ]:
data_mean, data_std = df_credit['V10'].mean(), df_credit['V10'].std()
cut_off = data_std * 3
lower, upper = data_mean - cut_off, data_mean + cut_off

print('The lower bound value is:', data_mean - cut_off)
print('The upper bound value is:', data_mean + cut_off)

plt.figure(figsize = (10,5))
sns.histplot(x = 'V10', data=df_credit, bins=70)
plt.axvspan(xmin = lower,xmax= df_credit.V10.min(),alpha=0.2, color='red')
plt.axvspan(xmin = upper,xmax= df_credit.V10.max(),alpha=0.2, color='red')
plt.show()


In [ ]:
def Metodo_StDev(df, n, features, threshold=3):
    """Devuelve índices marcados en al menos n variables por Z-score clásico."""
    
    return None

In [ ]:
outliers_StDev = Metodo_StDev(df_credit,1,df_credit.columns) #Indices que representan outliers

In [ ]:
outliers_StDev

In [ ]:
df_credit_wout = df_credit.drop(outliers_StDev, axis = 0).reset_index(drop=True)
df_credit_wout.head()

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=3,figsize=(13,8))
fig.suptitle('Algunas distribuciones después de aplicar el método de desviación estandar\n', size = 18)

axes[0,0].hist(df_credit_wout['V17'], bins=60, linewidth=0.5, edgecolor="black")
axes[0,0].set_title("V17");

axes[0,1].hist(df_credit_wout['V10'], bins=60, linewidth=0.5, edgecolor="black")
axes[0,1].set_title("V10");

axes[0,2].hist(df_credit_wout['V12'], bins=60, linewidth=0.5, edgecolor="black")
axes[0,2].set_title("V12");

axes[1,0].hist(df_credit_wout['V16'], bins=60, linewidth=0.5, edgecolor="black")
axes[1,0].set_title("V16");

axes[1,1].hist(df_credit_wout['V14'], bins=60, linewidth=0.5, edgecolor="black")
axes[1,1].set_title("V14");

axes[1,2].hist(df_credit_wout['V3'], bins=60, linewidth=0.5, edgecolor="black")
axes[1,2].set_title("V3");

axes[2,0].hist(df_credit_wout['V7'], bins=60, linewidth=0.5, edgecolor="black")
axes[2,0].set_title("V7");

axes[2,1].hist(df_credit_wout['V11'], bins=60, linewidth=0.5, edgecolor="black")
axes[2,1].set_title("V11");

axes[2,2].hist(df_credit_wout['V4'], bins=60, linewidth=0.5, edgecolor="black")
axes[2,2].set_title("V4");

plt.tight_layout()
plt.show()

### Método de puntuación Z (Z-score)

Al calcular el **Z-Score** (puntuación Z) se utiliza para convertir los datos en otro conjunto con media = 0, describe la posición de un valor bruto en términos de su distancia respecto a la media, medida en unidades de desviación estándar.

Esta técnica asume una **distribución gaussiana** de los datos. Los valores atípicos serán los datos que están en las colas de la distribución. En la mayoría de los casos se utiliza un umbral de 3 o -3, es decir, si el valor de la puntuación Z es mayor o menor que 3 o -3 respectivamente, ese punto de datos será identificado como valor atípico.

In [ ]:
df_taxis = sns.load_dataset('taxis')
df_taxis.head()

In [ ]:
# Usando un método de visualización (histograma)
plt.figure(figsize=(5,4))
sns.histplot(data=df_taxis, x = 'total', bins=60, linewidth=0.5, edgecolor="black")
plt.show()

In [ ]:
z_score =  (df_taxis['total'] - df_taxis['total'].mean())/df_taxis['total'].std()
z_score

In [ ]:
plt.figure(figsize=(5,4))
sns.histplot(data=z_score, bins=60, linewidth=0.5, edgecolor="black")
plt.title("Datos transformados por z-score")
plt.show()

In [ ]:
z_score_abs = abs(z_score)
outliers_zscore = z_score_abs > 3 #+- 3 veces
outliers_zscore

In [ ]:
df_taxis[~outliers_zscore]['total'] 

In [ ]:
plt.figure(figsize=(5,4))
sns.histplot(data=df_taxis[~outliers_zscore]['total'], bins=60, linewidth=0.5, edgecolor="black")
plt.show()

In [ ]:
def Metodo_Z_score (df,n,features):
    
    
    return None

In [ ]:
df_taxis.head(1)

In [ ]:
df_taxis.columns

In [ ]:
df_taxis.select_dtypes(include='number').head(1) # Columnas numericas

In [ ]:
lista_caracteristicas = df_taxis.select_dtypes(include='number').columns.to_list()
lista_caracteristicas

In [ ]:
outliers_z_score = Metodo_Z_score(df_taxis,1,lista_caracteristicas)

# dropping outliers
df_woutliers_taxi = df_taxis.drop(outliers_z_score, axis = 0).reset_index(drop=True)

In [ ]:
df_woutliers_taxi.head()

In [ ]:
from scipy.stats import zscore

z_scores = zscore(df_taxis['total'])
abs_z_scores = np.abs(z_scores)
df_taxis[abs_z_scores >3]['total']

La elección de 3 como umbral proviene de la regla empírica, según la cual los datos dentro de 3 veces la desviación estándar respecto a la media representan el 99.7% de los datos de la distribución. Sabiendo esto, podemos concluir con bastante seguridad que los datos que caen más allá de este umbral son atípicos, pues son distintos al 99.7% de los datos.

###  Z-Score Modificado

Cuando los datos son asimétricos o no se distribuyen de forma normal podemos utilizar el **z-score modificado (MAD-Z Score)**. el z-score modificado mide cuánto se aleja un valor de la mediana en términos de la desviación absoluta mediana.

$$ M_i = \frac{0.6745*(x_i - Mediana)}{MAD}$$

donde:
- $x_i$: Un valor de dato individual
- $Mediana$: La mediana del conjunto de datos
- $MAD$: La desviación absoluta mediana del conjunto de datos

La desviación absoluta mediana ($MAD$) es una estadística robusta de variabilidad que mide la dispersión de un conjunto de datos. Es menos afectada por valores atípicos que otras medidas de dispersión como la desviación estándar y la varianza. 

Si los datos son normales, la desviación estándar suele ser la mejor opción para evaluar la dispersión. Sin embargo, si los datos no son normales, el MAD es una estadística que puedes usar en su lugar.

$$MAD = Mediana(|x_i – x_m|)$$

donde:
- $x_i$: El i-ésimo valor en el conjunto de datos
- $x_m$: El valor mediano en el conjunto de datos

**Ejemplo:** Considere los datos $(1, 1, 2, 2 , 4, 6, 9)$. Su mediana es $2$. Las desviaciones absolutas con respecto a $2$ son $(1, 1, 0, 0, 2, 4, 7)$, que a su vez tienen una mediana de $1$ (ya que las desviaciones absolutas ordenadas son $(0, 0, 1, 1 , 2, 4, 7)$). Por lo tanto, la desviación absoluta mediana de estos datos es $1$.

In [ ]:
df_taxis.head(1)

In [ ]:
median = df_taxis['total'].median() #caluclando la mediana
median

In [ ]:
abs_diff = (df_taxis['total'] - median).abs() #desviaciones absolutas con respecto a la mediana
abs_diff.median()

In [ ]:
from scipy.stats import median_abs_deviation #Usanndo la librería de python
mad_score = median_abs_deviation(df_taxis['total'])
mad_score

In [ ]:
def Metodo_Z_ScoreMod(df, n, features, threshold=3):
    
    return None

In [ ]:
# detectando outliers mediante el método z-score modificado
outliers_z_score = Metodo_Z_ScoreMod(df_taxis,2,lista_caracteristicas)

# eliminando los outliers del dataframe
df_out_zscoremod_taxis = df_taxis.drop(outliers_z_score, axis = 0).reset_index(drop=True)
df_out_zscoremod_taxis.head()

In [ ]:
df_taxis.loc[outliers_z_score]

In [ ]:
len(df_out_zscoremod_taxis), len(df_taxis)

In [ ]:
#!pip install pyod
from pyod.models.mad import MAD
mad = MAD(threshold = 3)
lab = mad.fit(df_taxis['total'].values.reshape(-1,1)).labels_
lab

In [ ]:
sum(lab)

In [ ]:
df_taxis_woutliers = df_taxis[lab == 0]
df_taxis_woutliers.shape

In [ ]:
df_taxis_woutliers.head()

In [ ]:
df_taxis[lab!=0]

## Práctica de Laboratorio

1. Usa IQR para detectar candidatos en `sepal width (cm)`. ¿Cuántos encuentras y qué observaciones conservarías tras revisarlas?
2. Compara el número de candidatos producido por IQR, Z-score clásico y MAD-Z en `df_taxis['total']`. ¿Por qué no tienen que coincidir?
3. Explica por qué un umbral de `3` desviaciones estándar es menos confiable cuando la distribución es muy asimétrica.
5. En el ejemplo de regresión, compara la pendiente con todos los puntos contra la pendiente sin el punto atípico. ¿Qué significa que cambie mucho?
